In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path
import json 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.gridspec as gridspec
import geopandas as gpd

import torch
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

BARPLOT_ALPHA = 0.7

In [ ]:
def to_roman(n):
    vals = [(1000, 'm'), (900, 'cm'), (500, 'd'), (400, 'cd'), (100, 'c'),
            (90, 'xc'), (50, 'l'), (40, 'xl'), (10, 'x'), (9, 'ix'), (5, 'v'),
            (4, 'iv'), (1, 'i')]
    result = ''
    for v, s in vals:
        while n >= v:
            result += s
            n -= v
    return result

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
print(num_measurements)

SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

### Subregions:

In [ ]:
# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

In [ ]:
# With subregions
TARGET_REGIONS_SUB = [
    'IT',
    'AT',
    'SJM',
    'ALA',
    'CA_3',
    'CA_4',
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

## Model assets:

#### Pooled set: 
Pool source regions together

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["df_train"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses
glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

# Failsafe: check that pooled training data only contains source region codes
actual_codes = set(df_all_glaciers.region.unique())
expected_codes = set(SOURCE_REGIONS)
unexpected = actual_codes - expected_codes
if unexpected:
    raise ValueError(
        f"Pooled training set contains unexpected SOURCE_CODEs: {unexpected}")
else:
    print(f"✓ Pooled set SOURCE_CODEs OK: {actual_codes}")

print(f"Pooled model assets: {len(assets_full['ds_train'])} sequences")

#### Test (hold-out) regions:

In [ ]:
# ── Always compute scalers/blurs — needed for any recomputation ───────────────
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["df_train"],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_joint, scaler_geo = build_global_scalers_multi_source_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

blur_m, blur_s, blur_joint, blur_geo = estimate_global_bandwidths_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

In [ ]:
RECOMPUTE_SPLITS = False

FT_FRAC = 0.10
save_path = BASE_DIR / "glacier_splits_FD.json"

#gl_level_split = ['IT', 'AT']
gl_level_split = []
id_level_split = ['IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4']

# ── Load existing glacier splits from disk ────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        splits = json.load(f)
    print(f"Loaded splits from: {save_path}")

# ── Only compute glacier splits for regions that need them ────────────────────
regions_needing_gl_split = [r for r in gl_level_split]
regions_to_compute = (regions_needing_gl_split if RECOMPUTE_SPLITS else
                      [r for r in regions_needing_gl_split if r not in splits])

for region in regions_to_compute:
    print(f"\n{'='*50}\nSplitting region: {region}")
    split = split_pool_holdout_sinkhorn(
        df_region=monthly_cache[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split
    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

if regions_to_compute:
    with open(save_path, "w") as f:
        json.dump(splits, f, indent=2)
    print(f"\nSaved splits for: {regions_to_compute}")

In [ ]:
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = []

N_REPEATS = 5

# ── Build FT assets for all target subregions, repeated N_REPEATS times ──────
ft_assets_glacier = {}
ft_assets_id = {region: [] for region in TARGET_REGIONS_SUB}

for region in TARGET_REGIONS_SUB:
    print(f"\n{'='*50}\nBuilding FT assets for: {region}")

    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]
    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # 1. Glacier-level split (unchanged — deterministic by glacier, no repeats needed)
    if region in gl_level_split:
        if region in gl_level_split:
            pool_glaciers = splits[region]["pool_glaciers"]
            holdout_glaciers = splits[region]["holdout_glaciers"]
            exp_key = f"FD_to_{region}"

            ds_ft_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_FT",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    pool_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_TEST",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(
                    holdout_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    holdout_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)
            ft_assets_glacier[region] = {
                "ds_ft": ds_ft_gl,
                "ds_test": ds_test_gl,
                "ft_train_idx": ft_train_idx_gl,
                "ft_val_idx": ft_val_idx_gl,
            }
            print(
                f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} seqs | ds_test: {len(ds_test_gl)} seqs"
            )

    # 2. ID-level split, repeated N_REPEATS times with different seeds
    if region in id_level_split:
        winter_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "winter"]["ID"].unique().tolist()
        annual_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "annual"]["ID"].unique().tolist()

        for rep in range(N_REPEATS):
            winter_ids = list(winter_ids_full)
            annual_ids = list(annual_ids_full)

            rep_seed = cfg.seed + rep
            rng = np.random.default_rng(rep_seed)
            rng.shuffle(winter_ids)
            rng.shuffle(annual_ids)

            n_ft_w = max(1, int(len(winter_ids) *
                                FT_FRAC)) if winter_ids else 0
            n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
            ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
            test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

            # keep track of ft_ids per rep for the cross-rep check below
            if rep == 0:
                _ft_id_sets_for_check = []
            _ft_id_sets_for_check.append(set(ft_ids))

            exp_key_id = f"FD_to_{region}_IDsplit_rep{rep}"
            ds_ft_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_FT",
                df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_TEST",
                df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_id), val_ratio=0.2, seed=rep_seed)

            ft_assets_id[region].append({
                "rep": rep,
                "seed": rep_seed,
                "ds_ft": ds_ft_id,
                "ds_test": ds_test_id,
                "ft_train_idx": ft_train_idx_id,
                "ft_val_idx": ft_val_idx_id,
            })
            print(f"  [ID-split rep {rep}] ds_ft  : {len(ds_ft_id)} seqs | "
                  f"ds_test: {len(ds_test_id)} seqs")

            # ── sanity check after all reps for this region are built ────────────
        if region in id_level_split:
            n_reps = len(_ft_id_sets_for_check)
            identical_pairs = [
                (i, j) for i in range(n_reps) for j in range(i + 1, n_reps)
                if _ft_id_sets_for_check[i] == _ft_id_sets_for_check[j]
            ]
            if identical_pairs:
                print(
                    f"  WARNING [{region}]: identical FT id-sets for repeats {identical_pairs}"
                )
            else:
                overlaps = [
                    len(_ft_id_sets_for_check[i] & _ft_id_sets_for_check[j]) /
                    len(_ft_id_sets_for_check[i]) for i in range(n_reps)
                    for j in range(i + 1, n_reps)
                ]
                print(
                    f"  [check] {region}: {n_reps} repeats distinct (mean overlap {np.mean(overlaps):.1%})"
                )

# convenience dict: glacier-split where available, ID-split (list of reps) for the rest
ft_assets = {
    **ft_assets_glacier,
    **{
        r: ft_assets_id[r]
        for r in ft_assets_id if r not in ft_assets_glacier
    },
}

## Transformer:

### Zero shot:

In [ ]:
def _add_nrmse(metrics, preds_df):
    """Add NRMSE (normalized by std of observed PMB) for annual and winter."""
    preds_clean = preds_df.dropna(subset=["target", "pred"])

    std_annual = preds_clean.loc[preds_clean["PERIOD"] == "annual",
                                 "target"].std()
    std_winter = preds_clean.loc[preds_clean["PERIOD"] == "winter",
                                 "target"].std()

    metrics["NRMSE_annual"] = (metrics["RMSE_annual"] / std_annual
                               if std_annual and std_annual > 0 else np.nan)
    metrics["NRMSE_winter"] = (metrics["RMSE_winter"] / std_winter
                               if std_winter and std_winter > 0 else np.nan)
    return metrics

#### Pooled model:

In [ ]:
TRAIN_FLAG = False
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_pooled, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=PARAMS_TRANSFORMER,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Pooled model saved to: {model_path}")

# Build scaler once outside the loop
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

print(f"\n{'='*60}")
print(
    "Zero-shot evaluation of pooled model on target regions (full region test set)"
)
print(f"{'='*60}")

# def _eval_full_region(region):
#     ds_test_full = build_or_load_lstm_dataset_only(
#         cfg=cfg,
#         key=f"ZEROSHOT_{region}_FULL",
#         df_loss=monthly_cache[region]["data_monthly"],
#         df_full=monthly_cache[region]["data_monthly_aug"],
#         months_head_pad=monthly_cache[region]["months_head_pad"],
#         months_tail_pad=monthly_cache[region]["months_tail_pad"],
#         MONTHLY_COLS=MONTHLY_COLS,
#         STATIC_COLS=STATIC_COLS,
#         cache_dir=str(CACHE_DIR),
#         force_recompute=False,
#         kind="test",
#         show_progress=False,
#     )
#     ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
#         ds_test_full)
#     test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
#         ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
#     metrics, preds_df = model_pooled.evaluate_with_preds(device, test_dl, ds_test_copy)
#     metrics = _add_nrmse(metrics, preds_df)
#     return metrics

# # ── glacier-split: single dict per region, unchanged ──────────────────────────
# print(f"\n--- glacier-split ---")
# zs_metrics_full = {}
# for region in gl_level_split:
#     metrics = _eval_full_region(region)
#     zs_metrics_full[region] = metrics
#     print(f"  {region:15s}  RMSE_a={metrics['RMSE_annual']:.3f}  "
#           f"NRMSE_a={metrics['NRMSE_annual']:.3f}  "
#           f"RMSE_w={metrics['RMSE_winter']:.3f}  "
#           f"NRMSE_w={metrics['NRMSE_winter']:.3f}  "
#           f"R2_a={metrics['R2_annual']:.3f}  "
#           f"R2_w={metrics['R2_winter']:.3f}")

# # ── ID-split: evaluate once on the full target region, not per rep ────────────
# print(f"\n--- ID-split (evaluated on full target region, single value) ---")
# for region in id_level_split:
#     metrics = _eval_full_region(region)
#     zs_metrics_full[region] = metrics
#     print(f"  {region:15s}  RMSE_a={metrics['RMSE_annual']:.3f}  "
#           f"NRMSE_a={metrics['NRMSE_annual']:.3f}  "
#           f"RMSE_w={metrics['RMSE_winter']:.3f}  "
#           f"NRMSE_w={metrics['NRMSE_winter']:.3f}  "
#           f"R2_a={metrics['R2_annual']:.3f}  "
#           f"R2_w={metrics['R2_winter']:.3f}")

#### Single-region models:

In [ ]:
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION/models"
TRAIN_FLAG = False

single_source_models = {}
single_source_assets = {}
single_source_scalers = {}

for src in SOURCE_REGIONS:
    ckpt = XREG_MODELS_BASE / src / f"transformer_xreg_{src}_{src}_{MODEL_DATE}.pt"
    os.makedirs(ckpt.parent, exist_ok=True)

    df_src = res_xreg_by_source[src]["df_train"][["GLACIER"]] \
               .drop_duplicates().assign(region=src) \
               .rename(columns={"GLACIER": "glacier"})
    assets_src = build_assets_from_glacier_list(
        glaciers=df_src["glacier"].tolist(),
        df_ranked=df_src,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(XREG_MODELS_BASE / src / f"assets_{src}_100pct.joblib"),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=[src],
    )
    single_source_assets[src] = assets_src

    # build per-source scaler
    scaler_src = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_src["ds_train"])
    scaler_src.make_loaders(
        train_idx=assets_src["train_idx"],
        val_idx=assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    single_source_scalers[src] = scaler_src

    m = mbm.models.Transformer_MB.build_model_from_params(cfg,
                                                          PARAMS_TRANSFORMER,
                                                          device,
                                                          verbose=False)
    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state)
        print(f"Loaded {src} model from {ckpt}")
    else:
        print(f"No checkpoint found for {src}, training from scratch...")
        m, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=src,
            lstm_assets=assets_src,
            best_params=PARAMS_TRANSFORMER,
            device=device,
            models_dir=XREG_MODELS_BASE / src,
            prefix="transformer_xreg",
            train_flag=TRAIN_FLAG,
            force_retrain=False,
            epochs=N_EPOCHS,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        print(f"Trained {src} model, saved to {ckpt}")

    single_source_models[src] = m

#### Combine metrics:

In [ ]:
eval_configs = []
for region in TARGET_REGIONS_SUB:
    ds_test_full = build_or_load_lstm_dataset_only(
        cfg=cfg,
        key=f"ZEROSHOT_{region}_FULL",
        df_loss=monthly_cache[region]["data_monthly"],
        df_full=monthly_cache[region]["data_monthly_aug"],
        months_head_pad=monthly_cache[region]["months_head_pad"],
        months_tail_pad=monthly_cache[region]["months_tail_pad"],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=str(CACHE_DIR),
        force_recompute=False,
        kind="test",
        show_progress=False,
    )
    eval_configs.append((region, {"ds_test": ds_test_full}))


def _eval_on(model, ds_test, scaler):
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_test)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, scaler, batch_size=128, seed=cfg.seed)
    return model.evaluate_with_preds(device, test_dl, ds_copy)


df_metrics_comparison = {}
region_pred_target_ranges = {
}  # {region: {model_label: {"y_min":, "y_max":, "pred_min":, "pred_max":}}}

for tgt_label, assets in eval_configs:
    if assets is None:
        print(f"No assets for {tgt_label}, skipping.")
        continue

    df_metrics_comparison[tgt_label] = {}
    region_pred_target_ranges[tgt_label] = {}

    m, preds_df = _eval_on(model_pooled, assets["ds_test"], ds_pooled_scaler)
    m = _add_nrmse(m, preds_df)
    df_metrics_comparison[tgt_label]["pooled_zs"] = m
    preds_clean = preds_df.dropna(subset=["target", "pred"])
    region_pred_target_ranges[tgt_label]["pooled_zs"] = {
        "y_min": float(preds_clean["target"].min()),
        "y_max": float(preds_clean["target"].max()),
        "pred_min": float(preds_clean["pred"].min()),
        "pred_max": float(preds_clean["pred"].max()),
    }

    for src, model in single_source_models.items():
        m, preds_df = _eval_on(model, assets["ds_test"],
                               single_source_scalers[src])
        m = _add_nrmse(m, preds_df)
        df_metrics_comparison[tgt_label][src] = m
        preds_clean = preds_df.dropna(subset=["target", "pred"])
        region_pred_target_ranges[tgt_label][src] = {
            "y_min": float(preds_clean["target"].min()),
            "y_max": float(preds_clean["target"].max()),
            "pred_min": float(preds_clean["pred"].min()),
            "pred_max": float(preds_clean["pred"].max()),
        }

# --- print summary ---
print(f"\n{'='*90}")
print(
    f"{'Model':<15} {'RMSE_a':>8} {'NRMSE_a':>8} {'RMSE_w':>8} {'NRMSE_w':>8} {'R2_a':>8} {'R2_w':>8}"
)
print(f"{'='*90}")
for tgt_label, results in df_metrics_comparison.items():
    print(f"\n--- {tgt_label} ---")
    for model_label, m in results.items():
        rng = region_pred_target_ranges[tgt_label][model_label]
        print(f"  {model_label:<15}  "
              f"RMSE_a={m['RMSE_annual']:.3f}  "
              f"NRMSE_a={m['NRMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  "
              f"NRMSE_w={m['NRMSE_winter']:.3f}  "
              f"R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}  "
              f"y∈[{rng['y_min']:.2f}, {rng['y_max']:.2f}]  "
              f"pred∈[{rng['pred_min']:.2f}, {rng['pred_max']:.2f}]")

#### Barplot:

##### Annual:

In [ ]:
# glacier counts per region, from the data overview table
REGION_GLACIER_COUNTS = {
    "ALA": 14,
    "ISL": 55,
    "SJM": 3,
    "NOR": 32,
    "CH": 30,
    "FR": 9,
    "IT": 14,
    "AT": 10,
    "CA_1": 3,
    "CA_2": 2,
    "CA_3": 8,
    "CA_4": 2,
}

In [ ]:
# --- bar plot ---
MODEL_ORDER = ["pooled_zs"] + SOURCE_REGIONS
LABELS = {src: f"{src} only" for src in SOURCE_REGIONS}
LABELS["pooled_zs"] = "Multi-region"
METRICS = [
    ("RMSE_annual", "RMSE annual (m w.e.)", False),
]

MAX_COLS = 3

EXCLUDE_REGIONS = {
    'CENTRALASIA',
}
tgt_labels = [
    t for t in df_metrics_comparison.keys() if t not in EXCLUDE_REGIONS
]
n_tgts = len(tgt_labels)
n_metrics = len(METRICS)
n_cols = min(MAX_COLS, n_tgts)
n_rows_per_metric = math.ceil(n_tgts / n_cols)
n_rows = n_metrics * n_rows_per_metric

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.5 * n_cols, 3.8 * n_rows),
    sharey=True,
)
axes = np.array(axes).reshape(n_rows, n_cols)

label_fontsize = NATURE_SPECS["font_max_pt"] - 2
for metric_row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
    for tgt_idx, tgt_label in enumerate(tgt_labels):
        sub_row = tgt_idx // n_cols
        col = tgt_idx % n_cols
        row = metric_row * n_rows_per_metric + sub_row
        ax = axes[row, col]

        ax.text(0.03,
                0.97,
                f"({chr(ord('a') + tgt_idx)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left",
                zorder=5)

        results = df_metrics_comparison[tgt_label]
        models = [m for m in MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [COLORS[m] for m in models]

        bars = ax.bar(range(len(models)),
                      vals,
                      color=colors,
                      edgecolor="white",
                      linewidth=0.8,
                      alpha=0.5,
                      zorder=3)

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)

        for i, mod in enumerate(models):
            if mod.startswith("pooled_zs"):
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val in zip(bars, vals):
            y_pos = bar.get_height() + 0.02 if val >= 0 else bar.get_height(
            ) - 0.08
            ax.text(bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    f"{val:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=9)

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels([LABELS[m] for m in models],
                           rotation=90,
                           ha="right",
                           fontsize=label_fontsize)

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=label_fontsize)
        ax.tick_params(axis="y", labelsize=NATURE_SPECS["font_min_pt"] - 2)
        valid_vals = [v for v in vals if not np.isnan(v)]
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_vals) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_vals) * 1.25
        ax.set_ylim(ymin, ymax)

        n_glaciers = REGION_GLACIER_COUNTS.get(tgt_label)
        title_str = "Target: {}".format(REGION_LABELS.get(
            tgt_label, tgt_label))
        if n_glaciers is not None:
            title_str += f" ({n_glaciers} glaciers)"

        ax.set_title(title_str,
                     fontsize=NATURE_SPECS["font_max_pt"] - 2,
                     pad=6)

# Hide any unused axes in the last row
for empty_idx in range(n_tgts, n_rows_per_metric * n_cols):
    sub_row = empty_idx // n_cols
    col = empty_idx % n_cols
    for metric_row in range(n_metrics):
        axes[metric_row * n_rows_per_metric + sub_row, col].set_visible(False)

fig.suptitle("Multi-region versus single-region zero-shot transfer",
             fontsize=NATURE_SPECS["font_max_pt"] + 2)
fig.tight_layout(pad=1.8)
fig.subplots_adjust(top=0.9)
fig.savefig("figures/paperTF/fig5_zs_pooled_vs_ss_barplot_a.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

##### Winter:

In [ ]:
# --- bar plot ---
MODEL_ORDER = ["pooled_zs"] + SOURCE_REGIONS
LABELS = {src: f"{src} only" for src in SOURCE_REGIONS}
LABELS["pooled_zs"] = "Multi-region"
METRICS = [
    ("RMSE_winter", "RMSE winter (m w.e.)", False),
]

MAX_COLS = 3

EXCLUDE_REGIONS = {
    'CENTRALASIA',
}
tgt_labels = [
    t for t in df_metrics_comparison.keys() if t not in EXCLUDE_REGIONS
]
n_tgts = len(tgt_labels)
n_metrics = len(METRICS)
n_cols = min(MAX_COLS, n_tgts)
n_rows_per_metric = math.ceil(n_tgts / n_cols)
n_rows = n_metrics * n_rows_per_metric

# ── pre-compute one global ylim per metric (shared across ALL panels) ─────────
shared_ylims = {}
for metric_key, _, is_r2 in METRICS:
    all_vals = []
    for tgt_label in tgt_labels:
        results = df_metrics_comparison[tgt_label]
        models = [m for m in MODEL_ORDER if m in results]
        all_vals += [
            results[m][metric_key] for m in models
            if not np.isnan(results[m][metric_key])
        ]
    if not all_vals:
        continue
    if is_r2:
        ymin = min(min(all_vals) * 1.3, -0.2)
        ymax = max(max(all_vals) * 1.3, 0.5)
    else:
        ymin = 0
        ymax = max(all_vals) * 1.25
    shared_ylims[metric_key] = (ymin, ymax)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.5 * n_cols, 3.8 * n_rows),
    sharey=False,
)
axes = np.array(axes).reshape(n_rows, n_cols)

label_fontsize = NATURE_SPECS["font_max_pt"] - 2
for metric_row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
    for tgt_idx, tgt_label in enumerate(tgt_labels):
        sub_row = tgt_idx // n_cols
        col = tgt_idx % n_cols
        row = metric_row * n_rows_per_metric + sub_row
        ax = axes[row, col]

        ax.text(0.03,
                0.97,
                f"({chr(ord('a') + tgt_idx)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left",
                zorder=5)

        results = df_metrics_comparison[tgt_label]
        models = [m for m in MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [COLORS[m] for m in models]

        bars = ax.bar(range(len(models)),
                      vals,
                      color=colors,
                      edgecolor="white",
                      linewidth=0.8,
                      alpha=0.5,
                      zorder=3)

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)

        for i, mod in enumerate(models):
            if mod.startswith("pooled_zs"):
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val in zip(bars, vals):
            if np.isnan(val):
                continue
            y_pos = bar.get_height() + 0.02 if val >= 0 else bar.get_height(
            ) - 0.08
            ax.text(bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    f"{val:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=9)

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels([LABELS[m] for m in models],
                           rotation=90,
                           ha="right",
                           fontsize=label_fontsize)

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=label_fontsize)
        ax.tick_params(axis="y", labelsize=NATURE_SPECS["font_min_pt"] - 2)

        if metric_key in shared_ylims:
            ax.set_ylim(shared_ylims[metric_key])

        # hide redundant y tick labels on inner columns
        if col != 0:
            ax.tick_params(labelleft=False)

        n_glaciers = REGION_GLACIER_COUNTS.get(tgt_label)
        title_str = "Target: {}".format(REGION_LABELS.get(
            tgt_label, tgt_label))
        if n_glaciers is not None:
            title_str += f" ({n_glaciers} glaciers)"

        ax.set_title(title_str,
                     fontsize=NATURE_SPECS["font_max_pt"] - 2,
                     pad=6)

# Hide any unused axes in the last row
for empty_idx in range(n_tgts, n_rows_per_metric * n_cols):
    sub_row = empty_idx // n_cols
    col = empty_idx % n_cols
    for metric_row in range(n_metrics):
        axes[metric_row * n_rows_per_metric + sub_row, col].set_visible(False)

fig.suptitle(
    "Multi-region versus single-region zero-shot transfer: winter mass balance",
    fontsize=NATURE_SPECS["font_max_pt"] + 2)
fig.tight_layout(pad=1.8)
fig.subplots_adjust(top=0.9)
fig.savefig("figures/paperTF/appendix/A5_zs_pooled_vs_ss_barplot_w.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

#### Scatter:

In [ ]:
def _eval_and_plot(model,
                   ds_test,
                   scaler,
                   ax,
                   lim,
                   legend_fontsize=NATURE_SPECS["font_min_pt"]):
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_test)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, scaler, batch_size=128, seed=cfg.seed)
    metrics, test_df_preds = model.evaluate_with_preds(device, test_dl,
                                                       ds_copy)
    test_df_preds = test_df_preds.dropna(subset=["target", "pred"])
    scores_annual, scores_winter = compute_seasonal_scores(test_df_preds,
                                                           target_col="target",
                                                           pred_col="pred")

    pred_vs_truth_density(
        ax,
        test_df_preds,
        scores_annual,
        add_legend=False,
        palette=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
        ax_xlim=lim,
        ax_ylim=lim,
        s=50,
    )

    def _fmt(x):
        return "NA" if (x is None or
                        (isinstance(x, float) and np.isnan(x))) else f"{x:.2f}"

    legend_NN = "\n".join([
        rf"$\mathrm{{RMSE_a}}={_fmt(scores_annual['rmse'])},\ \mathrm{{RMSE_w}}={_fmt(scores_winter['rmse'])}$",
        rf"$\mathrm{{R^2_a}}={_fmt(scores_annual['R2'])},\ \mathrm{{R^2_w}}={_fmt(scores_winter['R2'])}$",
        rf"$\mathrm{{Bias_a}}={_fmt(scores_annual['Bias'])},\ \mathrm{{Bias_w}}={_fmt(scores_winter['Bias'])}$",
    ])
    ax.text(0.98,
            0.02,
            legend_NN,
            transform=ax.transAxes,
            va="bottom",
            ha="right",
            fontsize=legend_fontsize,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.5))
    return metrics


EXCLUDE_REGIONS = {'CENTRALASIA'}
regions_to_plot = [
    t for t in df_metrics_comparison.keys() if t not in EXCLUDE_REGIONS
]

plot_model_order = ["pooled_zs"] + SOURCE_REGIONS
COL_TITLES_ZS = {
    "pooled_zs": "Multi-region",
    **{
        src: f"Single-region: {REGION_LABELS.get(src, src)}"
        for src in SOURCE_REGIONS
    },
}
model_and_scaler = {
    "pooled_zs": (model_pooled, ds_pooled_scaler),
    **{
        src: (single_source_models[src], single_source_scalers[src])
        for src in SOURCE_REGIONS
    },
}

ncols = len(plot_model_order)
nrows = len(regions_to_plot)

region_lims_zs = {}
for region in regions_to_plot:
    ds_test = dict(eval_configs)[region]["ds_test"]
    vals = ds_test.y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.5
    region_lims_zs[region] = (float(vals.min() - pad), float(vals.max() + pad))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 4 * nrows),
    sharex="row",
    sharey="row",
)
axes = np.array(axes).reshape(nrows, ncols)

eval_configs_dict = dict(eval_configs)

for row_i, region in enumerate(regions_to_plot):
    lim = region_lims_zs[region]
    ds_test = eval_configs_dict[region]["ds_test"]

    for col_i, model_key in enumerate(plot_model_order):
        ax = axes[row_i, col_i]
        model, scaler = model_and_scaler[model_key]

        _eval_and_plot(model,
                       ds_test,
                       scaler,
                       ax,
                       lim,
                       legend_fontsize=NATURE_SPECS["font_min_pt"])
        apply_nature_style(ax,
                           fontsize=NATURE_SPECS["font_max_pt"] + 2,
                           box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        row_letter = chr(ord('a') + row_i)
        panel_label = f"{row_letter}{col_i + 1}"
        ax.text(
            0.02,
            0.98,
            f"({panel_label})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            #fontweight="bold",
            va="top",
            ha="left")

        if row_i == 0:
            ax.set_title(
                COL_TITLES_ZS.get(model_key, model_key),
                fontsize=NATURE_SPECS["font_max_pt"] + 5,
            )
        if col_i == 0:
            ax.set_ylabel(
                f"Target: {REGION_LABELS.get(region, region)}\nModeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"] + 2)
        else:
            ax.set_ylabel("")
        if row_i < nrows - 1:
            ax.set_xlabel("")

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"] + 5,
)
fig.suptitle(
    "Multi-region vs single-region zero-shot transfer",
    fontsize=NATURE_SPECS["font_max_pt"] + 6,
    #fontweight="bold",
)
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.subplots_adjust(top=0.94)
fig.savefig("figures/paperTF/appendix/app_zs_pooled_vs_ss_scatter_all.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

### Fine tuning:

#### Simple fine tuning:

In [ ]:
models_dir_ft = BASE_DIR / "models" / "finetuned"
os.makedirs(models_dir_ft, exist_ok=True)

ft_models_gl = {}  # glacier-split
ft_models_id = {}  # ID-split, now nested by rep

# ── glacier-split: unchanged structure ────────────────────────────────────────
for region in gl_level_split:
    ft_models_gl[region] = {}
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region} (GLsplit)")

    for strategy in ["head_only", "full", "adapter"]:
        ft_models_gl[region][strategy] = finetune_transformer_on_target(
            cfg=cfg,
            model_pooled=model_pooled,
            ds_ft=ft_assets_glacier[region]["ds_ft"],
            ds_pooled_scaler=ds_pooled_scaler,
            device=device,
            best_params=PARAMS_TRANSFORMER,
            model_filename=str(
                models_dir_ft /
                f"transformer_ft_{strategy}_{region}_{MODEL_DATE}.pt"),
            strategy=strategy,
            epochs=PARAMS_FT['epochs'],
            lr_factor=PARAMS_FT['lr_factor'],
            force_retrain=False,
        )
        print(f"  [{region}] {strategy} done")

# ── ID-split: loop over reps, one model set per rep ──────────────────────────
for region in id_level_split:
    ft_models_id[region] = []  # list of per-rep strategy dicts
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region} (IDsplit, {N_REPEATS} repeats)")

    for rep_data in ft_assets_id[region]:
        rep = rep_data["rep"]
        rep_models = {}

        for strategy in ["head_only", "full"]:
            rep_models[strategy] = finetune_transformer_on_target(
                cfg=cfg,
                model_pooled=model_pooled,
                ds_ft=rep_data["ds_ft"],
                ds_pooled_scaler=ds_pooled_scaler,
                device=device,
                best_params=PARAMS_TRANSFORMER,
                model_filename=str(
                    models_dir_ft /
                    f"transformer_ft_{strategy}_{region}_IDsplit_rep{rep}_{MODEL_DATE}.pt"
                ),
                strategy=strategy,
                epochs=PARAMS_FT['epochs'],
                lr_factor=PARAMS_FT['lr_factor'],
                force_retrain=False,
            )
            print(f"  [{region}] rep {rep} - {strategy} done")

        ft_models_id[region].append({"rep": rep, "models": rep_models})

#### DAN fine tuning:

In [ ]:
# source_codes_pretrain = build_source_codes_for_dataset(
#     assets_full["ds_train"],
#     pd.concat(
#         [res_xreg_by_source[r]["df_train"] for r in SOURCE_REGIONS],
#         ignore_index=True,
#     ),
#     source_col="SOURCE_CODE",
# )

# dan_models = {}  # glacier-split
# dan_models_id = {}  # ID-split, now nested by rep

# for region in id_level_split:
#     print(f"\n{'='*50}  DAN → {region} (IDsplit, {N_REPEATS} repeats)")

#     dan_models_id[region] = []  # list of per-rep DAN models

#     for rep_data in ft_assets_id[region]:
#         rep = rep_data["rep"]

#         source_codes_ft = [region] * len(rep_data["ds_ft"])

#         dan_model, _ = finetune_dan_transformer_on_target(
#             cfg=cfg,
#             model_foundation=model_pooled,
#             assets_full=assets_full,
#             ft_assets_region=rep_data,
#             ds_pooled_scaler=ds_pooled_scaler,
#             source_codes_pretrain=source_codes_pretrain,
#             source_codes_ft=source_codes_ft,
#             device=device,
#             best_params=PARAMS_TRANSFORMER,
#             model_filename=str(
#                 models_dir_ft /
#                 f"transformer_dan_{region}_IDsplit_rep{rep}_{MODEL_DATE}.pt"),
#             dan_alpha=PARAMS_DAN['dan_alpha'],
#             grl_lambda=PARAMS_DAN['grl_lambda'],
#             mix_ratio_ft=PARAMS_DAN['mix_ratio_ft'],
#             epochs=PARAMS_DAN['epochs'],
#             lr_backbone=PARAMS_DAN['lr_backbone'],
#             lr_domain=PARAMS_DAN['lr_domain'],
#             force_retrain=False,
#             verbose=False,
#         )

#         dan_models_id[region].append({"rep": rep, "model": dan_model})
#         print(f"  [{region}] rep {rep} DAN done")

In [ ]:
# --- unified evaluation --- ID-split only, aggregated over repeats
print(f"\n{'='*95}")
print(
    f"{'Strategy':<25} {'RMSE_a':>8} {'NRMSE_a':>8} {'RMSE_w':>8} {'NRMSE_w':>8} {'R2_a':>8} {'R2_w':>8}"
)
print(f"{'='*95}")

df_ft_metrics_id = {}  # {region: {strategy_label: {metric: mean_value}}}
df_ft_metrics_id_raw = {
}  # {region: {strategy_label: [metrics_per_rep]}} for std/error bars

for region in id_level_split:
    print(f"\n--- {region} (mean over {N_REPEATS} repeats) ---")

    df_ft_metrics_id_raw[region] = {
        label: []
        for label in ["no_ft", "head_only", "full"]
    }

    for rep_idx, rep_data in enumerate(ft_assets_id[region]):
        rep = rep_data["rep"]

        def _eval(model, ds_test=rep_data["ds_test"]):
            ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                ds_test)
            test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
            metrics, preds_df = model.evaluate_with_preds(
                device, test_dl, ds_test_copy)
            return _add_nrmse(metrics, preds_df)

        rep_ft_models = ft_models_id[region][rep_idx]["models"]
        #rep_dan_model = dan_models_id[region][rep_idx]["model"]

        all_models = {
            "no_ft": model_pooled,
            "head_only": rep_ft_models["head_only"],
            #"adapter": rep_ft_models["adapter"],
            "full": rep_ft_models["full"],
            #"dan": rep_dan_model,
        }

        for label, model in all_models.items():
            m = _eval(model)
            df_ft_metrics_id_raw[region][label].append(m)

    # aggregate across reps
    df_ft_metrics_id[region] = {}
    for label, rep_metrics_list in df_ft_metrics_id_raw[region].items():
        agg = {}
        for metric_key in rep_metrics_list[0].keys():
            vals = [m[metric_key] for m in rep_metrics_list]
            agg[metric_key] = float(np.mean(vals))
            agg[f"{metric_key}_std"] = float(np.std(vals))
        df_ft_metrics_id[region][label] = agg

        print(
            f"  {label:<25}  "
            f"RMSE_a={agg['RMSE_annual']:.3f}±{agg['RMSE_annual_std']:.3f}  "
            f"NRMSE_a={agg['NRMSE_annual']:.3f}±{agg['NRMSE_annual_std']:.3f}  "
            f"RMSE_w={agg['RMSE_winter']:.3f}±{agg['RMSE_winter_std']:.3f}  "
            f"NRMSE_w={agg['NRMSE_winter']:.3f}±{agg['NRMSE_winter_std']:.3f}  "
            f"R2_a={agg['R2_annual']:.3f}±{agg['R2_annual_std']:.3f}  "
            f"R2_w={agg['R2_winter']:.3f}±{agg['R2_winter_std']:.3f}")

#### Scatter plot:

In [ ]:
import matplotlib.gridspec as gridspec

EXCLUDE_REGIONS = {
    'CENTRALASIA',
}
regions_to_plot = [r for r in id_level_split if r not in EXCLUDE_REGIONS]
strategies = ["head_only", "full"]

PLOT_REP = 0

COL_TITLES = {
    "no_ft (zero-shot)": "Zero-shot",
    "head_only": "Head only FT",
    "full": "Full FT",
}

plot_configs_labels = ["no_ft (zero-shot)"] + strategies
n_rows = len(regions_to_plot)
n_cols = len(plot_configs_labels)

# Precompute per-region axis limits
region_lims = {}
for region in regions_to_plot:
    rep_data = ft_assets_id[region][PLOT_REP]
    vals = rep_data["ds_test"].y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.5
    region_lims[region] = (float(vals.min() - pad), float(vals.max() + pad))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4 * n_cols, 3.4 * n_rows),
    squeeze=False,
)

# Column titles: strategy names
for col_i, label in enumerate(plot_configs_labels):
    axes[0, col_i].set_title(
        COL_TITLES.get(label, label),
        fontsize=NATURE_SPECS["font_max_pt"] + 1,
    )

for region_idx, region in enumerate(regions_to_plot):
    lim = region_lims[region]
    rep_data = ft_assets_id[region][PLOT_REP]
    rep_ft_models = next(d["models"] for d in ft_models_id[region]
                         if d["rep"] == PLOT_REP)

    plot_configs = [
        ("no_ft (zero-shot)", model_pooled, assets_full),
        *[(strategy, rep_ft_models[strategy], assets_full)
          for strategy in strategies],
    ]

    for col_i, (label, model, assets_train) in enumerate(plot_configs):
        ax = axes[region_idx, col_i]

        evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key={
                "ds_train": assets_train["ds_train"],
                "ds_test": rep_data["ds_test"],
                "train_idx": assets_train["train_idx"],
                "val_idx": assets_train["val_idx"],
            },
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        panel_label = chr(ord('a') + region_idx) + str(col_i + 1)
        ax.text(0.02,
                0.98,
                f"({panel_label})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left")

        # Region name on left y-axis
        if col_i == 0:
            ax.set_ylabel(
                f"Target: {REGION_LABELS.get(region, region)}\nModeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelleft=False)

        # x-axis labels only on bottom row
        if region_idx < n_rows - 1:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelbottom=False)

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.005),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    "Effect of fine-tuning compared to zero-shot transfer performance (10% local data)",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
)
fig.tight_layout()
fig.subplots_adjust(top=0.94, bottom=0.08)
fig.savefig("figures/paperTF/appendix/app_ft_strategies_all.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec

EXCLUDE_REGIONS = {
    'CENTRALASIA',
}
regions_to_plot = [r for r in id_level_split if r not in EXCLUDE_REGIONS]
strategies = ["full"]

PLOT_REP = 0

COL_TITLES = {
    "no_ft (zero-shot)": "Zero-shot",
    "head_only": "Head only FT",
    "full": "Full FT",
}

plot_configs_labels = ["no_ft (zero-shot)"] + strategies


# Precompute per-region axis limits
region_lims = {}
for region in regions_to_plot:
    rep_data = ft_assets_id[region][PLOT_REP]
    vals = rep_data["ds_test"].y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.5
    region_lims[region] = (float(vals.min() - pad), float(vals.max() + pad))



# Column titles: strategy names
for col_i, label in enumerate(plot_configs_labels):
    axes[0, col_i].set_title(
        COL_TITLES.get(label, label),
        fontsize=NATURE_SPECS["font_max_pt"] + 1,
    )

#regions_to_plot = ['IT', 'SJM', 'ALA', 'CA_3']
regions_to_plot = ['IT', 'ALA']
n_rows = len(regions_to_plot)
n_cols = len(plot_configs_labels)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4 * n_cols, 3.4 * n_rows),
    squeeze=False,
)


for region_idx, region in enumerate(regions_to_plot):
    lim = region_lims[region]
    rep_data = ft_assets_id[region][PLOT_REP]
    rep_ft_models = next(d["models"] for d in ft_models_id[region]
                         if d["rep"] == PLOT_REP)

    plot_configs = [
        ("no_ft (zero-shot)", model_pooled, assets_full),
        *[(strategy, rep_ft_models[strategy], assets_full)
          for strategy in strategies],
    ]

    for col_i, (label, model, assets_train) in enumerate(plot_configs):
        ax = axes[region_idx, col_i]

        evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key={
                "ds_train": assets_train["ds_train"],
                "ds_test": rep_data["ds_test"],
                "train_idx": assets_train["train_idx"],
                "val_idx": assets_train["val_idx"],
            },
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        panel_label = chr(ord('a') + region_idx) + str(col_i + 1)
        # ax.text(0.02,
        #         0.98,
        #         f"({panel_label})",
        #         transform=ax.transAxes,
        #         fontsize=NATURE_SPECS["font_max_pt"],
        #         va="top",
        #         ha="left")

        # Region name on left y-axis
        if col_i == 0:
            ax.set_ylabel(
                #f"Target: {REGION_LABELS.get(region, region)}\nModeled PMB (m w.e.)",
                f"Modeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelleft=False)

        # x-axis labels only on bottom row
        if region_idx < n_rows - 1:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelbottom=False)

# fig.legend(
#     handles=[
#         mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
#         mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
#     ],
#     loc="lower center",
#     bbox_to_anchor=(0.5, 0.005),
#     ncols=2,
#     frameon=False,
#     fontsize=NATURE_SPECS["font_max_pt"],
# )
# fig.suptitle(
#     "Effect of fine-tuning compared to zero-shot transfer performance (10% local data)",
#     fontsize=NATURE_SPECS["font_max_pt"] + 2,
# )
fig.tight_layout()
fig.subplots_adjust(top=0.94, bottom=0.08)
fig.savefig("figures/paperTF/appendix/app_ft_strategies_ppt_1.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.gridspec as gridspec

EXCLUDE_REGIONS = {
    'CENTRALASIA',
}
regions_to_plot = [r for r in id_level_split if r not in EXCLUDE_REGIONS]
strategies = ["full"]

PLOT_REP = 0

COL_TITLES = {
    "no_ft (zero-shot)": "Zero-shot",
    "head_only": "Head only FT",
    "full": "Full FT",
}

plot_configs_labels = ["no_ft (zero-shot)"] + strategies


# Precompute per-region axis limits
region_lims = {}
for region in regions_to_plot:
    rep_data = ft_assets_id[region][PLOT_REP]
    vals = rep_data["ds_test"].y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.5
    region_lims[region] = (float(vals.min() - pad), float(vals.max() + pad))



# Column titles: strategy names
for col_i, label in enumerate(plot_configs_labels):
    axes[0, col_i].set_title(
        COL_TITLES.get(label, label),
        fontsize=NATURE_SPECS["font_max_pt"] + 1,
    )

#regions_to_plot = ['IT', 'SJM', 'ALA', 'CA_3']
regions_to_plot = ['SJM', 'CA_3']
n_rows = len(regions_to_plot)
n_cols = len(plot_configs_labels)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4 * n_cols, 3.4 * n_rows),
    squeeze=False,
)


for region_idx, region in enumerate(regions_to_plot):
    lim = region_lims[region]
    rep_data = ft_assets_id[region][PLOT_REP]
    rep_ft_models = next(d["models"] for d in ft_models_id[region]
                         if d["rep"] == PLOT_REP)

    plot_configs = [
        ("no_ft (zero-shot)", model_pooled, assets_full),
        *[(strategy, rep_ft_models[strategy], assets_full)
          for strategy in strategies],
    ]

    for col_i, (label, model, assets_train) in enumerate(plot_configs):
        ax = axes[region_idx, col_i]

        evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key={
                "ds_train": assets_train["ds_train"],
                "ds_test": rep_data["ds_test"],
                "train_idx": assets_train["train_idx"],
                "val_idx": assets_train["val_idx"],
            },
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        panel_label = chr(ord('c') + region_idx) + str(col_i + 1)
        # ax.text(0.02,
        #         0.98,
        #         f"({panel_label})",
        #         transform=ax.transAxes,
        #         fontsize=NATURE_SPECS["font_max_pt"],
        #         va="top",
        #         ha="left")

        # Region name on left y-axis
        if col_i == 0:
            ax.set_ylabel(
                #f"Target: {REGION_LABELS.get(region, region)}\nModeled PMB (m w.e.)",
                f"Modeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelleft=False)

        # x-axis labels only on bottom row
        if region_idx < n_rows - 1:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelbottom=False)

# fig.legend(
#     handles=[
#         mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
#         mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
#     ],
#     loc="lower center",
#     bbox_to_anchor=(0.5, 0.005),
#     ncols=2,
#     frameon=False,
#     fontsize=NATURE_SPECS["font_max_pt"],
# )
# fig.suptitle(
#     "Effect of fine-tuning compared to zero-shot transfer performance (10% local data)",
#     fontsize=NATURE_SPECS["font_max_pt"] + 2,
# )
fig.tight_layout()
fig.subplots_adjust(top=0.94, bottom=0.08)
fig.savefig("figures/paperTF/appendix/app_ft_strategies_ppt_2.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()



#### Bar plot:

##### Annual:

In [ ]:
# ── Fine-tuning gain barplot ──────────────────────────────────────────────────

FT_MODEL_ORDER = ["no_ft", "head_only", "full"]
FT_LABELS = {
    "no_ft": "Zero-shot",
    "head_only": "Head only FT",
    "full": "Full FT",
    "dan": "DAN",
}
FT_COLORS = {
    "no_ft": "#aaaaaa",
    "head_only": "#EE854A",
    "dan": "#D65F5F",
    "full": "#956CB4",
}

METRICS_FT = [
    ("RMSE_annual", "RMSE annual (m w.e.)", False),
]

MAX_COLS_FT = 3
EXCLUDE_FT = {"CENTRALASIA"}

tgt_labels_ft = [t for t in df_ft_metrics_id.keys() if t not in EXCLUDE_FT]

n_tgts_ft = len(tgt_labels_ft)
n_metrics_ft = len(METRICS_FT)
n_cols_ft = min(MAX_COLS_FT, n_tgts_ft)
n_col_groups = math.ceil(n_tgts_ft / n_cols_ft)

PLOT_H = 3
SPACER_H = 1.3

height_ratios = []
for g in range(n_col_groups):
    height_ratios.append(PLOT_H)
    if g < n_col_groups - 1:
        height_ratios.append(SPACER_H)

total_rows = len(height_ratios)
fig_h = sum(height_ratios) + 0.8

fig = plt.figure(figsize=(5 * n_cols_ft, fig_h))
gs = GridSpec(
    total_rows,
    n_cols_ft,
    figure=fig,
    height_ratios=height_ratios,
    hspace=0.24,
    wspace=0.15,  # tighter, since inner y tick labels are hidden
)


def gs_row(group, metric_offset):
    return group * 2


# ── pre-compute one global ylim per metric (shared across ALL panels) ─────────
shared_ylims = {}
for tgt_idx, tgt_label in enumerate(tgt_labels_ft):
    results = df_ft_metrics_id[tgt_label]
    models = [m for m in FT_MODEL_ORDER if m in results]

    for metric_offset, (metric_key, _, is_r2) in enumerate(METRICS_FT):
        std_key = f"{metric_key}_std"
        vals = [results[m][metric_key] for m in models]
        stds = [results[m].get(std_key, 0.0) for m in models]
        valid_pairs = [(v, s) for v, s in zip(vals, stds) if not np.isnan(v)]
        if not valid_pairs:
            continue
        valid_vals = [v for v, _ in valid_pairs]
        valid_tops = [v + s for v, s in valid_pairs]  # account for error bar extent
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_tops) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_tops) * 1.25
        key = metric_offset
        if key not in shared_ylims:
            shared_ylims[key] = (ymin, ymax)
        else:
            shared_ylims[key] = (
                min(shared_ylims[key][0], ymin),
                max(shared_ylims[key][1], ymax),
            )

# ── plot ──────────────────────────────────────────────────────────────────────
for tgt_idx, tgt_label in enumerate(tgt_labels_ft):
    col = tgt_idx % n_cols_ft
    group_row = tgt_idx // n_cols_ft

    results = df_ft_metrics_id[tgt_label]
    models = [m for m in FT_MODEL_ORDER if m in results]
    colors = [FT_COLORS[m] for m in models]

    for metric_offset, (metric_key, metric_label,
                        is_r2) in enumerate(METRICS_FT):
        std_key = f"{metric_key}_std"
        row = gs_row(group_row, metric_offset)
        ax = fig.add_subplot(gs[row, col])

        panel_idx = group_row * n_cols_ft * n_metrics_ft + metric_offset * n_cols_ft + col
        ax.text(0.03,
                0.97,
                f"({chr(ord('a') + panel_idx)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left",
                zorder=5)

        vals = [results[m][metric_key] for m in models]
        stds = [results[m].get(std_key, 0.0) for m in models]

        bars = ax.bar(
            range(len(models)),
            vals,
            yerr=stds,
            capsize=3,
            error_kw={
                "linewidth": 1.0,
                "ecolor": "black",
                "zorder": 4
            },
            color=colors,
            edgecolor="white",
            linewidth=0.8,
            alpha=BARPLOT_ALPHA,
            zorder=3,
        )

        for i, mod in enumerate(models):
            if mod == "no_ft":
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val, std in zip(bars, vals, stds):
            if np.isnan(val):
                continue
            y_pos = bar.get_height(
            ) + std + 0.02 if val >= 0 else bar.get_height() - std - 0.08
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                y_pos,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        ax.set_xticks(range(len(models)))
        if metric_offset == n_metrics_ft - 1:
            ax.set_xticklabels(
                [FT_LABELS[m] for m in models],
                rotation=90,
                ha="right",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_max_pt"])

        if metric_offset == 0:
            n_glaciers = REGION_GLACIER_COUNTS.get(tgt_label)
            title_str = "Target: {}".format(
                REGION_LABELS.get(tgt_label, tgt_label))
            if n_glaciers is not None:
                title_str += f" ({n_glaciers} glaciers)"

            ax.set_title(title_str,
                         fontsize=NATURE_SPECS["font_max_pt"] - 4,
                         pad=6)

        key = metric_offset
        if key in shared_ylims:
            ax.set_ylim(shared_ylims[key])

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=False)

        # hide redundant y tick labels (after styling so they stay hidden)
        if col != 0:
            ax.tick_params(labelleft=False)

fig.suptitle(
    "Effect of fine-tuning compared to zero-shot transfer performance (10% local data)",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    y=1.0,
)
fig.savefig(
    "figures/paperTF/fig6_pooled_ft_gain_barplot.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

### Sensitivity to the fine-tuning fraction (reviewer response)

Fine-tuning fractions $f \in \{1, 5, 10, 25, 50\}\,\%$ of the target-region measurement IDs, `N_REPEATS_SENS` random repeats each, head-only and full fine-tuning.

**Design**
- Each repeat uses the **same seed and ID shuffle as the main experiment** (`cfg.seed + rep`; winter and annual IDs shuffled separately). The 10% subset is therefore identical to the FT set used in the main results.
- The **test set is fixed across fractions**: the IDs remaining after the largest (50%) FT subset. FT subsets are **nested** (1% ⊂ 5% ⊂ 10% ⊂ 25% ⊂ 50%). Differences between fractions come from the amount of FT data only, not from a changing test set. (Absolute RMSEs at 10% can differ slightly from the main table, because the main table evaluates on the remaining 90%.)
- Fine-tuning follows the same procedure as `finetune_transformer_on_target`: an 80/20 train/val split of the FT data, weighted sampler, AdamW with `lr × lr_factor`, ReduceLROnPlateau, early stopping (patience 10), and restoring the **best-validation checkpoint**. In addition, the per-epoch train/val loss is stored (epoch 0 = pooled model before fine-tuning).
- Runs whose FT set has fewer than `MIN_FT_SEQS` sequences are skipped (the internal validation split would be degenerate). These show up as missing points.


In [ ]:
import copy
import math


def finetune_with_history(
    cfg,
    model_pooled,
    ds_ft,
    ds_pooled_scaler,
    device,
    best_params,
    model_filename,
    strategy="full",
    epochs=50,
    lr_factor=0.1,
    split_seed=None,
    force_retrain=False,
    verbose=False,
):
    """
    Same fine-tuning procedure as `finetune_transformer_on_target` (same split,
    sampler, optimiser, scheduler, early stopping, best-val checkpoint), but it
    also returns and caches the per-epoch train/val loss history.

    The history includes epoch 0 (pooled model before any fine-tuning step),
    evaluated in eval mode on the same train/val loaders.
    Cached next to the checkpoint as `<model_filename>.history.json`.
    """
    assert strategy in ("head_only", "full")
    model_filename = str(model_filename)
    hist_path = Path(model_filename).with_suffix(".history.json")

    model_ft = copy.deepcopy(model_pooled)

    if (not force_retrain) and os.path.exists(model_filename) and hist_path.exists():
        model_ft.load_state_dict(torch.load(model_filename, map_location=device))
        with open(hist_path, "r") as f:
            history = json.load(f)
        return model_ft, history

    split_seed = cfg.seed if split_seed is None else split_seed
    train_idx, val_idx = mbm.data_processing.MBSequenceDataset.split_indices(
        len(ds_ft), val_ratio=0.2, seed=split_seed)

    ds_ft_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(ds_ft)
    ds_ft_copy.set_scalers_from(ds_pooled_scaler)
    ds_ft_copy.transform_inplace()
    train_dl, val_dl = ds_ft_copy.make_loaders(
        train_idx=train_idx,
        val_idx=val_idx,
        fit_and_transform=False,
        seed=cfg.seed,
        use_weighted_sampler=True,
        verbose=False,
    )

    # --- freeze / unfreeze (identical to finetune_transformer_on_target) ---
    if strategy == "head_only":
        for p in model_ft.parameters():
            p.requires_grad = False
        head_modules = ([model_ft.head_w, model_ft.head_a]
                        if model_ft.two_heads else [model_ft.head])
        for m in head_modules:
            for p in m.parameters():
                p.requires_grad = True

    n_trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model_ft.parameters())

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model_ft.parameters()),
        lr=best_params["lr"] * lr_factor,
        weight_decay=best_params["weight_decay"],
    )
    loss_fn = mbm.models.Transformer_MB.resolve_loss_fn(best_params)

    # --- epoch 0: loss of the pooled model before fine-tuning ---
    tr0 = model_ft.run_epoch(device, None, train_dl, loss_fn=loss_fn, train=False)
    va0 = model_ft.run_epoch(device, None, val_dl, loss_fn=loss_fn, train=False)

    hist, best_val, _ = model_ft.train_loop(
        device=device,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=epochs,
        optimizer=optimizer,
        loss_fn=loss_fn,
        es_patience=10,
        es_min_delta=1e-4,
        sched_factor=0.5,
        sched_patience=4,
        sched_min_lr=1e-7,
        log_every=5,
        verbose=verbose,
        save_best_path=model_filename,
    )

    val_curve = hist["val_loss"]
    # epoch index (1-based, epoch 0 = before FT) of the restored checkpoint
    best_epoch = int(np.argmin(np.abs(np.array(val_curve) - best_val))) + 1

    history = {
        "train_loss": [float(tr0)] + [float(x) for x in hist["train_loss"]],
        "val_loss": [float(va0)] + [float(x) for x in val_curve],
        "lr": [float("nan")] + [float(x) for x in hist["lr"]],
        "best_epoch": best_epoch,
        "best_val_loss": float(best_val),
        "n_train": int(len(train_idx)),
        "n_val": int(len(val_idx)),
        "n_trainable_params": int(n_trainable),
        "n_total_params": int(n_total),
        "strategy": strategy,
    }
    with open(hist_path, "w") as f:
        json.dump(history, f)

    # train_loop already restored the best state; reload from disk to be explicit
    model_ft.load_state_dict(torch.load(model_filename, map_location=device))
    return model_ft, history


In [ ]:
# ── FT-fraction sensitivity: configuration & nested splits ───────────────────
FT_FRACS_SENS = [0.05, 0.10, 0.15, 0.2, 0.25, 0.3, 0.50]
N_REPEATS_SENS = N_REPEATS  # same seeds as main experiment
STRATEGIES_SENS = ["head_only", "full"]
REGIONS_SENS = list(id_level_split)
MIN_FT_SEQS = 5  # skip runs with fewer FT sequences (degenerate 80/20 split)
FORCE_RECOMPUTE_SENS_DATA = False

SENS_DIR = BASE_DIR / "ft_fraction_sensitivity"
models_dir_sens = SENS_DIR / "models"
os.makedirs(models_dir_sens, exist_ok=True)

FT_FRAC_MAX = max(FT_FRACS_SENS)


def _frac_tag(f):
    return f"{int(round(f * 100)):02d}pct"


def _n_ft(n_ids, frac):
    # identical rule to the main experiment
    return max(1, int(n_ids * frac)) if n_ids else 0


sens_assets = {r: {f: [] for f in FT_FRACS_SENS} for r in REGIONS_SENS}
sens_test = {r: [] for r in REGIONS_SENS}
sens_sizes = []

for region in REGIONS_SENS:
    print(f"\n{'='*50}\nFT-fraction sensitivity splits: {region}")
    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]

    winter_ids_full = df_monthly[df_monthly["PERIOD"] == "winter"]["ID"].unique().tolist()
    annual_ids_full = df_monthly[df_monthly["PERIOD"] == "annual"]["ID"].unique().tolist()

    for rep in range(N_REPEATS_SENS):
        # identical shuffle to the main experiment -> 10% subset == main FT set
        winter_ids = list(winter_ids_full)
        annual_ids = list(annual_ids_full)
        rep_seed = cfg.seed + rep
        rng = np.random.default_rng(rep_seed)
        rng.shuffle(winter_ids)
        rng.shuffle(annual_ids)

        # fixed test set: everything after the largest FT subset
        n_w_max = _n_ft(len(winter_ids), FT_FRAC_MAX)
        n_a_max = _n_ft(len(annual_ids), FT_FRAC_MAX)
        test_ids = winter_ids[n_w_max:] + annual_ids[n_a_max:]

        ds_test = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"FD_to_{region}_IDsplit_rep{rep}_FTSENS_TEST",
            df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
            df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=FORCE_RECOMPUTE_SENS_DATA,
            kind="test",
            show_progress=False,
        )
        sens_test[region].append({"rep": rep, "seed": rep_seed, "ds_test": ds_test,
                                  "test_ids": set(test_ids)})

        prev_ids = set()
        for frac in FT_FRACS_SENS:
            n_w = _n_ft(len(winter_ids), frac)
            n_a = _n_ft(len(annual_ids), frac)
            ft_ids = winter_ids[:n_w] + annual_ids[:n_a]

            # sanity: nested and disjoint from test
            assert not (set(ft_ids) & set(test_ids)), f"FT/test overlap {region} rep{rep} {frac}"
            assert prev_ids <= set(ft_ids), f"FT subsets not nested {region} rep{rep} {frac}"
            prev_ids = set(ft_ids)

            ds_ft = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"FD_to_{region}_IDsplit_rep{rep}_FTSENS_{_frac_tag(frac)}_FT",
                df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=FORCE_RECOMPUTE_SENS_DATA,
                kind="ft",
                show_progress=False,
            )
            sens_assets[region][frac].append({
                "rep": rep, "seed": rep_seed, "ds_ft": ds_ft,
                "n_ft_ids": len(ft_ids), "n_ft_w": n_w, "n_ft_a": n_a,
            })
            sens_sizes.append({
                "region": region, "rep": rep, "frac": frac,
                "n_ft_ids": len(ft_ids), "n_ft_winter": n_w, "n_ft_annual": n_a,
                "n_ft_seqs": len(ds_ft), "n_test_seqs": len(ds_test),
                "n_ft_glaciers": df_monthly[df_monthly["ID"].isin(ft_ids)]["GLACIER"].nunique(),
            })

        # optional: check that the 10% subset equals the main-experiment FT set
        if 0.10 in FT_FRACS_SENS and region in ft_assets_id and len(ft_assets_id[region]) > rep:
            n_main = len(ft_assets_id[region][rep]["ds_ft"])
            n_sens = len(sens_assets[region][0.10][rep]["ds_ft"])
            if n_main != n_sens:
                print(f"  WARNING [{region} rep{rep}]: 10% FT set size differs from main "
                      f"experiment ({n_sens} vs {n_main})")

df_sens_sizes = pd.DataFrame(sens_sizes)
print("\nFT set sizes (mean over repeats):")
display(
    df_sens_sizes.groupby(["region", "frac"])[["n_ft_seqs", "n_ft_glaciers", "n_test_seqs"]]
    .mean().round(1).unstack("frac"))


In [ ]:
# ── FT-fraction sensitivity: fine-tune + evaluate on fixed test set ──────────
FORCE_RETRAIN_SENS = False
SENS_RESULTS_CSV = SENS_DIR / f"ft_fraction_sensitivity_{MODEL_DATE}.csv"


def _eval_on_test(model, ds_test):
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(ds_test)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
    metrics, preds_df = model.evaluate_with_preds(device, test_dl, ds_copy)
    return _add_nrmse(metrics, preds_df)


def _scalar_metrics(m):
    return {k: float(v) for k, v in m.items() if np.isscalar(v)}


sens_rows = []
sens_histories = {}  # {(region, frac, strategy, rep): history dict}

for region in REGIONS_SENS:
    print(f"\n{'='*60}\nFT-fraction sensitivity: {region}")
    for rep in range(N_REPEATS_SENS):
        ds_test = sens_test[region][rep]["ds_test"]

        # zero-shot on the same fixed test set (reference line)
        m0 = _scalar_metrics(_eval_on_test(model_pooled, ds_test))
        sens_rows.append({"region": region, "rep": rep, "frac": 0.0,
                          "strategy": "no_ft", "n_ft_seqs": 0, **m0})

        for frac in FT_FRACS_SENS:
            a = sens_assets[region][frac][rep]
            n_ft = len(a["ds_ft"])
            if n_ft < MIN_FT_SEQS:
                print(f"  [{region} rep{rep} {_frac_tag(frac)}] skipped: "
                      f"only {n_ft} FT sequences (< {MIN_FT_SEQS})")
                continue

            for strategy in STRATEGIES_SENS:
                fname = (models_dir_sens /
                         f"transformer_ft_{strategy}_{region}_{_frac_tag(frac)}"
                         f"_rep{rep}_{MODEL_DATE}.pt")
                model_ft, hist = finetune_with_history(
                    cfg=cfg,
                    model_pooled=model_pooled,
                    ds_ft=a["ds_ft"],
                    ds_pooled_scaler=ds_pooled_scaler,
                    device=device,
                    best_params=PARAMS_TRANSFORMER,
                    model_filename=fname,
                    strategy=strategy,
                    epochs=PARAMS_FT["epochs"],
                    lr_factor=PARAMS_FT["lr_factor"],
                    force_retrain=FORCE_RETRAIN_SENS,
                    verbose=False,
                )
                sens_histories[(region, frac, strategy, rep)] = hist

                m = _scalar_metrics(_eval_on_test(model_ft, ds_test))
                be = hist["best_epoch"]
                sens_rows.append({
                    "region": region, "rep": rep, "frac": frac, "strategy": strategy,
                    "n_ft_seqs": n_ft, "n_ft_train": hist["n_train"], "n_ft_val": hist["n_val"],
                    "n_trainable_params": hist["n_trainable_params"],
                    "best_epoch": be,
                    "n_epochs_run": len(hist["train_loss"]) - 1,
                    "train_loss_ep0": hist["train_loss"][0],
                    "val_loss_ep0": hist["val_loss"][0],
                    "train_loss_best": hist["train_loss"][be],
                    "val_loss_best": hist["val_loss"][be],
                    "val_loss_final": hist["val_loss"][-1],
                    "train_loss_final": hist["train_loss"][-1],
                    **m,
                })
                del model_ft
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            print(f"  [{region} rep{rep} {_frac_tag(frac)}] n_ft={n_ft} done")

df_sens = pd.DataFrame(sens_rows)
# overfitting diagnostics
df_sens["gen_gap_best"] = df_sens["val_loss_best"] - df_sens["train_loss_best"]
df_sens["gen_ratio_best"] = df_sens["val_loss_best"] / df_sens["train_loss_best"]
df_sens["val_rise_after_best"] = df_sens["val_loss_final"] - df_sens["val_loss_best"]

df_sens.to_csv(SENS_RESULTS_CSV, index=False)
print(f"\nSaved per-run results to: {SENS_RESULTS_CSV}")


In [ ]:
# ── Summary tables ────────────────────────────────────────────────────────────
SUMMARY_METRICS = ["RMSE_annual", "RMSE_winter", "NRMSE_annual", "NRMSE_winter",
                   "R2_annual", "R2_winter"]

agg = (df_sens.groupby(["region", "strategy", "frac"])[SUMMARY_METRICS]
       .agg(["mean", "std"]))


def _fmt_pm(df, metric):
    return (df[(metric, "mean")].map("{:.3f}".format) + " ± " +
            df[(metric, "std")].map("{:.3f}".format))


for metric in ["RMSE_annual", "RMSE_winter"]:
    print(f"\n{metric} (mean ± std over {N_REPEATS_SENS} repeats, fixed test set)")
    tbl = _fmt_pm(agg, metric).unstack("frac")
    display(tbl)

# Gain over zero-shot, per repeat (paired, same test set)
zs = (df_sens[df_sens.strategy == "no_ft"]
      .set_index(["region", "rep"])[["RMSE_annual", "RMSE_winter"]])
df_ft_only = df_sens[df_sens.strategy != "no_ft"].copy()
for m in ["RMSE_annual", "RMSE_winter"]:
    df_ft_only[f"d{m}_vs_zs"] = (df_ft_only[m].values -
                                 zs.loc[list(zip(df_ft_only.region, df_ft_only.rep)), m].values)

print("\nChange in RMSE vs zero-shot (negative = improvement), mean over repeats")
display(df_ft_only.groupby(["region", "strategy", "frac"])
        [["dRMSE_annual_vs_zs", "dRMSE_winter_vs_zs"]].mean().round(3)
        .unstack("frac"))

# Full minus head-only, paired per repeat: is full FT ever worse at small fractions?
piv = df_ft_only.pivot_table(index=["region", "frac", "rep"], columns="strategy",
                             values=["RMSE_annual", "RMSE_winter"])
full_minus_head = pd.DataFrame({
    "RMSE_annual": piv[("RMSE_annual", "full")] - piv[("RMSE_annual", "head_only")],
    "RMSE_winter": piv[("RMSE_winter", "full")] - piv[("RMSE_winter", "head_only")],
})
print("\nFull FT − head-only FT RMSE (negative = full better); mean [frac. of repeats where full is better]")
fmh = full_minus_head.groupby(["region", "frac"]).agg(
    dRMSE_a=("RMSE_annual", "mean"),
    full_better_a=("RMSE_annual", lambda x: (x < 0).mean()),
    dRMSE_w=("RMSE_winter", "mean"),
    full_better_w=("RMSE_winter", lambda x: (x < 0).mean()),
).round(3)
display(fmh.unstack("frac"))

# Overfitting diagnostics from the loss curves
print("\nOverfitting diagnostics (mean over repeats): best epoch, epochs run, "
      "val−train loss gap at the restored checkpoint, val-loss rise after best epoch")
display(df_ft_only.groupby(["region", "strategy", "frac"])
        [["n_ft_train", "n_ft_val", "best_epoch", "n_epochs_run", "gen_gap_best",
          "gen_ratio_best", "val_rise_after_best"]].mean().round(3))


In [ ]:
# ── Sensitivity figure: RMSE vs fine-tuning fraction ─────────────────────────
_SENS_COLORS = {"no_ft": "#aaaaaa", "head_only": "#EE854A", "full": "#956CB4"}
_SENS_LABELS = {"no_ft": "Zero-shot", "head_only": "Head only FT", "full": "Full FT"}
try:  # reuse the colours/labels of the main FT barplot if defined
    _SENS_COLORS.update({k: v for k, v in FT_COLORS.items() if k in _SENS_COLORS})
    _SENS_LABELS.update({k: v for k, v in FT_LABELS.items() if k in _SENS_LABELS})
except NameError:
    pass

MAIN_FT_FRAC = FT_FRAC  # 0.10, highlighted in the plots
EXCLUDE_SENS_PLOT = {"CENTRALASIA"}
os.makedirs("figures/paperTF/appendix", exist_ok=True)


def plot_ft_fraction_sensitivity(df,
                                 metric,
                                 ylabel,
                                 regions,
                                 max_cols=3,
                                 xscale="linear",
                                 xtick_rotation=45,
                                 fname=None):
    regions = [r for r in regions if r not in EXCLUDE_SENS_PLOT]
    n_cols = min(max_cols, len(regions))
    n_rows = math.ceil(len(regions) / n_cols)
    fig, axes = plt.subplots(n_rows,
                             n_cols,
                             figsize=(3.6 * n_cols, 3.0 * n_rows),
                             squeeze=False,
                             sharex=True)
    xs_pct = [100 * f for f in FT_FRACS_SENS]
    fs = NATURE_SPECS["font_max_pt"]

    for i, region in enumerate(regions):
        ax = axes[i // n_cols, i % n_cols]
        d = df[df.region == region]

        # zero-shot reference band
        z = d[d.strategy == "no_ft"][metric]
        if len(z):
            ax.axhspan(z.mean() - z.std(), z.mean() + z.std(),
                       color=_SENS_COLORS["no_ft"], alpha=0.25, lw=0, zorder=1)
            ax.axhline(z.mean(), color=_SENS_COLORS["no_ft"], ls="-", lw=1.2,
                       label=_SENS_LABELS["no_ft"], zorder=2)

        for strategy in STRATEGIES_SENS:
            g = (d[d.strategy == strategy].groupby("frac")[metric]
                 .agg(["mean", "std"]).reindex(FT_FRACS_SENS))
            ax.errorbar(xs_pct, g["mean"], yerr=g["std"], marker="o", ms=4, lw=1.4,
                        capsize=2.5, color=_SENS_COLORS[strategy],
                        label=_SENS_LABELS[strategy], zorder=3)

        ax.axvline(100 * MAIN_FT_FRAC, color="k", ls="--", lw=0.8, zorder=0)
        ax.set_xscale(xscale)
        ax.set_xticks(xs_pct)
        ax.set_xticklabels([f"{x:g}" for x in xs_pct])
        ax.minorticks_off()

        # panel letter in the title -> never collides with data or axis labels
        ax.set_title(f"({chr(ord('a') + i)}) {REGION_LABELS.get(region, region)}",
                     fontsize=fs, loc="left")
        apply_nature_style(ax, fontsize=fs, box=True)

        is_bottom = (i // n_cols == n_rows - 1) or (i + n_cols >= len(regions))
        ax.tick_params(axis="x", labelbottom=is_bottom)
        if is_bottom:
            plt.setp(ax.get_xticklabels(), rotation=xtick_rotation,
                     ha="right" if xtick_rotation else "center",
                     rotation_mode="anchor")
            ax.set_xlabel("Fine-tuning fraction (%)", fontsize=fs)
        if i % n_cols == 0:
            ax.set_ylabel(ylabel, fontsize=fs)

    for j in range(len(regions), n_rows * n_cols):
        axes[j // n_cols, j % n_cols].set_visible(False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False,
               bbox_to_anchor=(0.5, 0.0), fontsize=fs)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    if fname:
        fig.savefig(fname, dpi=NATURE_SPECS["dpi"], bbox_inches="tight")
    plt.show()
    return fig


fig = plot_ft_fraction_sensitivity(
    df_sens, "RMSE_annual", "RMSE annual (m w.e.)", REGIONS_SENS,
    fname="figures/paperTF/appendix/app_ft_fraction_sensitivity_annual.png")
fig = plot_ft_fraction_sensitivity(
    df_sens, "RMSE_winter", "RMSE winter (m w.e.)", REGIONS_SENS,
    fname="figures/paperTF/appendix/app_ft_fraction_sensitivity_winter.png")

In [ ]:
# ── Training / validation loss curves ─────────────────────────────────────────
def plot_loss_grid(row_specs, strategies=STRATEGIES_SENS, logy=False, fname=None,
                   row_label_fn=None):
    """
    row_specs : list of (region, frac) tuples, one figure row each.
    One column per strategy. Thin lines = individual repeats
    (solid = validation, dashed = training); dot = restored best-val checkpoint.
    Epoch 0 = pooled model before fine-tuning.
    """
    n_rows, n_cols = len(row_specs), len(strategies)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 2.4 * n_rows),
                             squeeze=False, sharex=True)
    cmap = plt.get_cmap("viridis")

    for r_i, (region, frac) in enumerate(row_specs):
        for c_i, strategy in enumerate(strategies):
            ax = axes[r_i, c_i]
            n_plotted = 0
            for rep in range(N_REPEATS_SENS):
                h = sens_histories.get((region, frac, strategy, rep))
                if h is None:
                    continue
                col = cmap(rep / max(1, N_REPEATS_SENS - 1))
                ep = np.arange(len(h["train_loss"]))
                ax.plot(ep, h["train_loss"], ls="--", lw=0.9, color=col, alpha=0.8)
                ax.plot(ep, h["val_loss"], ls="-", lw=1.2, color=col, alpha=0.9)
                be = h["best_epoch"]
                ax.plot(be, h["val_loss"][be], "o", ms=4, color=col, mec="k", mew=0.5)
                n_plotted += 1

            if n_plotted == 0:
                ax.text(0.5, 0.5, f"skipped (< {MIN_FT_SEQS} FT seqs)", ha="center",
                        va="center", transform=ax.transAxes,
                        fontsize=NATURE_SPECS["font_min_pt"])
                ax.set_yticks([])
            if logy:
                ax.set_yscale("log")
            apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)

            if r_i == 0:
                ax.set_title(_SENS_LABELS[strategy], fontsize=NATURE_SPECS["font_max_pt"] + 1)
            if c_i == 0:
                lab = (row_label_fn(region, frac) if row_label_fn else
                       f"{REGION_LABELS.get(region, region)}, {100*frac:g}%")
                ax.set_ylabel(f"{lab}\nLoss", fontsize=NATURE_SPECS["font_max_pt"] - 1)
            if r_i == n_rows - 1:
                ax.set_xlabel("Epoch", fontsize=NATURE_SPECS["font_max_pt"])

            n_seq = (df_sens_sizes[(df_sens_sizes.region == region) &
                                   (df_sens_sizes.frac == frac)]["n_ft_seqs"].mean())
            ax.text(0.98, 0.97, f"n$_{{FT}}$={n_seq:.0f}", transform=ax.transAxes,
                    ha="right", va="top", fontsize=NATURE_SPECS["font_min_pt"] - 1)

    from matplotlib.lines import Line2D
    handles = [Line2D([], [], color="k", ls="--", lw=0.9, label="Training"),
               Line2D([], [], color="k", ls="-", lw=1.2, label="Validation"),
               Line2D([], [], color="w", marker="o", mfc="grey", mec="k", ms=5,
                      label="Restored checkpoint (min. val.)")]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False,
               bbox_to_anchor=(0.5, -0.01), fontsize=NATURE_SPECS["font_max_pt"])
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    if fname:
        fig.savefig(fname, dpi=NATURE_SPECS["dpi"], bbox_inches="tight")
    plt.show()
    return fig


# (1) All target regions at the fraction used in the paper (10%)
regions_plot = [r for r in REGIONS_SENS if r not in EXCLUDE_SENS_PLOT]
fig = plot_loss_grid(
    [(r, MAIN_FT_FRAC) for r in regions_plot],
    row_label_fn=lambda r, f: REGION_LABELS.get(r, r),
    fname="figures/paperTF/appendix/app_ft_loss_curves_10pct.png")

# (2) Svalbard (3 glaciers) across all fractions: the case raised by the reviewer
fig = plot_loss_grid(
    [("SJM", f) for f in FT_FRACS_SENS],
    row_label_fn=lambda r, f: f"{100*f:g}%",
    fname="figures/paperTF/appendix/app_ft_loss_curves_SJM_fractions.png")


## Distances:

In [ ]:
def add_cartesian_coords(df, lon_col="POINT_LON", lat_col="POINT_LAT"):
    """Add unit-sphere Cartesian coordinates to avoid longitude wraparound issues."""
    df = df.copy()
    lat_rad = np.radians(df[lat_col])
    lon_rad = np.radians(df[lon_col])
    df["COORD_X"] = np.cos(lat_rad) * np.cos(lon_rad)
    df["COORD_Y"] = np.cos(lat_rad) * np.sin(lon_rad)
    df["COORD_Z"] = np.sin(lat_rad)
    return df


COORD_COLS = ['COORD_X', 'COORD_Y', 'COORD_Z']
STATIC_COLS_WITH_COORDS = STATIC_COLS + COORD_COLS

# apply Cartesian projection to source-only training/test data used for scalers
res_source_only_coords = {
    k: {
        **v,
        "df_train":
        add_cartesian_coords(v["df_train"]),
        **({
            "df_test": add_cartesian_coords(v["df_test"])
        } if "df_test" in v else {}),
    }
    for k, v in res_source_only.items()
}

scaler_m, scaler_s, scaler_joint, scaler_geo = build_global_scalers_multi_source_simple(
    res_source_only_coords,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS_WITH_COORDS,
)
blur_m, blur_s, blur_joint, blur_geo = estimate_global_bandwidths_simple(
    res_source_only_coords,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS_WITH_COORDS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)

RECOMPUTE_SHIFTS_POOLED = False
SHIFTS_CACHE_POOLED = BASE_DIR / "domain_shifts_pooled_cache.pkl"

bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
scaler_joint = scaler_joint  # already computed in scalers cell

EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
tgt_labels = [t for t in df_ft_metrics_id.keys() if t not in EXCLUDE_REGIONS]

if RECOMPUTE_SHIFTS_POOLED and SHIFTS_CACHE_POOLED.exists():
    SHIFTS_CACHE_POOLED.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE_POOLED}")

if SHIFTS_CACHE_POOLED.exists():
    with open(SHIFTS_CACHE_POOLED, "rb") as f:
        cache = pickle.load(f)
    all_shifts_pooled = cache["all_shifts_pooled"]
    print(f"Loaded pooled shifts from cache: {SHIFTS_CACHE_POOLED}")

else:
    # df_src = full pooled training data (all source regions combined)
    df_src_pooled = pd.concat(
        [res_xreg_by_source[r]["df_train"] for r in SOURCE_REGIONS],
        ignore_index=True,
    )
    df_src_pooled = add_cartesian_coords(df_src_pooled)

    all_shifts_pooled = {}
    for region in tqdm(tgt_labels, desc="Pooled domain shifts"):
        df_tgt = add_cartesian_coords(monthly_cache[region]["data_monthly"])
        shift = compute_domain_shift(
            df_src=df_src_pooled,
            df_tgt=df_tgt,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS_WITH_COORDS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            scaler_joint=scaler_joint,
            blur_m=blur_m,
            blur_s=blur_s,
            blur_joint=blur_joint,
            bandwidths_m=bandwidths_m,
            bandwidths_s=bandwidths_s,
            seed=cfg.seed,
        )
        all_shifts_pooled[region] = shift
        print(f"  {region}: joint={shift['D_sinkhorn_joint']:.4f} "
              f"clim={shift['D_sinkhorn_climate']:.4f} "
              f"topo={shift['D_sinkhorn_topo']:.4f}")

    with open(SHIFTS_CACHE_POOLED, "wb") as f:
        pickle.dump({"all_shifts_pooled": all_shifts_pooled}, f)
    print(f"Saved pooled shifts to cache: {SHIFTS_CACHE_POOLED}")

# fig = plot_domain_shift_across_regions(all_shifts_pooled, src_region="Pooled")
# plt.show()

In [ ]:
from scipy import stats

TF_COLOR = NATURE_PALETTE["sky_blue"]


def plot_pooled_performance_vs_shift(
    all_shifts_pooled: dict,
    df_metrics: dict,
    metric: str = "RMSE_annual",
    model_label: str = "no_ft",  # zero-shot entry in df_ft_metrics_id
    distance_metrics: list[str] = [
        "D_sinkhorn_climate", "D_sinkhorn_topo", "D_sinkhorn_joint"
    ],
    model: str = "Transformer",
):
    # ── 1. Build merged DataFrame ─────────────────────────────────────────────
    shift_rows = []
    for tgt, shift in all_shifts_pooled.items():
        row = {"target": tgt}
        row.update(shift)
        shift_rows.append(row)
    df_shifts = pd.DataFrame(shift_rows)

    perf_rows = []
    for tgt, results in df_metrics.items():
        if model_label not in results:
            continue
        row = {"target": tgt, metric: results[model_label][metric]}
        std_key = f"{metric}_std"
        if std_key in results[model_label]:
            row[std_key] = results[model_label][std_key]
        perf_rows.append(row)
    df_perf = pd.DataFrame(perf_rows)

    df_plot = df_shifts.merge(df_perf, on="target")

    # ── 2. Plot ───────────────────────────────────────────────────────────────
    dist_labels = {
        "D_sinkhorn_climate": "Sinkhorn distance (climate)",
        "D_sinkhorn_topo": "Sinkhorn distance (topographic)",
        "D_sinkhorn_joint": "Sinkhorn distance (joint)",
        "D_mmd2_climate": "MMD² (climate)",
        "D_mmd2_topo": "MMD² (topographic)",
        "D_energy_climate": "Energy distance (climate)",
    }

    n = len(distance_metrics)
    fig, axes = plt.subplots(
        1,
        n,
        figsize=(4.5 * n, 4),
        constrained_layout=False,  # turn off auto-redistribution
    )
    fig.subplots_adjust(left=0.08,
                        right=0.98,
                        bottom=0.18,
                        top=0.85,
                        wspace=0.3)
    if n == 1:
        axes = [axes]

    std_key = f"{metric}_std"
    has_err = std_key in df_plot.columns

    for ax, dist_key in zip(axes, distance_metrics):
        x = df_plot[dist_key].values
        y = df_plot[metric].values
        labels = df_plot["target"].values
        yerr = df_plot[std_key].values if has_err else None

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            fmt="o",
            color=TF_COLOR,
            markersize=8,
            zorder=3,
            ecolor="0.5",
            elinewidth=1.0,
            capsize=3,
            markeredgecolor="white",
            markeredgewidth=0.5,
        )

        for xi, yi, lab in zip(x, y, labels):
            ax.annotate(
                f"Pooled \u2192 {lab}",
                (xi, yi),
                xytext=(5, 4),
                textcoords="offset points",
                fontsize=8,
                color="0.3",
            )

        # leave headroom above the highest point for its annotation
        y_min, y_max = y.min(), y.max()
        y_range = y_max - y_min if y_max > y_min else 1.0
        ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.5)

        # correlation annotation
        if len(x) > 2:
            r, p = stats.pearsonr(x, y)
            # ax.text(
            #     0.05, 0.95,
            #     f"r={r:.2f}, p={p:.3f}",
            #     transform=ax.transAxes,
            #     fontsize=8,
            #     va="top",
            #     ha="left",
            #     bbox=dict(boxstyle="round", facecolor="white", alpha=0.6),
            # )

        ax.set_xlabel(dist_labels.get(dist_key, dist_key), fontsize=9)
        ax.set_ylabel(
            f"{model} RMSE annual (m w.e.)"
            if dist_key == distance_metrics[0] else "",
            fontsize=9,
        )
        #ax.set_title(dist_labels.get(dist_key, dist_key), fontsize=10)

    #fig.suptitle(f"Pooled model: {model} RMSE vs. domain shift", fontsize=11)
    return fig

### KDE plots:

In [ ]:
def plot_kde_pair_reg(glaciers_to_plot, selected_cols):
    """KDE panels for a pair of glaciers."""
    ncols = 4
    nrows = int(np.ceil(len(selected_cols) / ncols))
    w, h = nature_figsize(cols=1, height_mm=160)
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * 2, h), squeeze=False)

    legend_handles = []

    for idx, col in enumerate(selected_cols):
        ax = axes[idx // ncols][idx % ncols]

        all_vals = pd.concat([
            cfg_gl["df"][col].dropna() for cfg_gl in glaciers_to_plot.values()
        ])
        x_grid = np.linspace(float(all_vals.min()), float(all_vals.max()), 500)

        for region_code, cfg_gl in glaciers_to_plot.items():
            vals = cfg_gl["df"][col].dropna().values
            if len(vals) < 10:
                continue
            color = COLORS.get(region_code, "#888888")
            label = REGION_LABELS.get(region_code, region_code)
            kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
            y = kde(x_grid)
            y = y / y.max()
            (line, ) = ax.plot(x_grid,
                               y,
                               color=color,
                               linewidth=0.8,
                               label=label)
            ax.fill_between(x_grid, y, alpha=0.08, color=color)

            # collect handles only from first panel to avoid duplicates
            if idx == 0:
                legend_handles.append(line)

        var_name = vois_climate_long_name.get(col, col)
        unit = vois_units.get(col, "")
        title = f"{var_name} ({unit})" if unit else var_name
        ax.set_title(title, fontsize=8)

        # panel label (a), (b), (c)...
        panel_label = chr(ord('a') + idx)
        ax.text(
            0.02,
            0.98,
            f"({panel_label})",
            transform=ax.transAxes,
            fontsize=8,
            va="top",
            ha="left",
            zorder=5,
        )

        ax.set_ylim(0, 1.15)
        ax.set_xlabel("")
        ax.tick_params(labelsize=8, width=0.4, length=2, direction="in")
        ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
        ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
        ax.set_axisbelow(True)
        format_axis_ticks(ax, label_size=6)

    # ── place legend in first empty axis if one exists, else above figure ──
    empty_axes = [
        axes[idx // ncols][idx % ncols]
        for idx in range(len(selected_cols), nrows * ncols)
    ]

    if empty_axes:
        leg_ax = empty_axes[0]
        leg_ax.axis("off")
        leg_ax.legend(
            handles=legend_handles,
            loc="center",
            fontsize=8,
            frameon=False,
        )
        # turn off remaining empty axes
        for ax in empty_axes[1:]:
            ax.axis("off")
    else:
        # no empty panels — place legend above the figure
        fig.legend(
            handles=legend_handles,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=len(glaciers_to_plot),
            fontsize=8,
            frameon=False,
        )

    plt.tight_layout(h_pad=3.0, rect=[0, 0.03, 1, 1])
    plt.show()
    return fig

In [ ]:
COLORS[
    "Pooled"] = "black"  # or "#aaaaaa" to match pooled_zs, or pick something new
REGION_LABELS["Pooled"] = "Pooled"

df_src_pooled = pd.concat(
    [res_xreg_by_source[r]["df_train"] for r in SOURCE_REGIONS],
    ignore_index=True)

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
] + ["POINT_LON", "POINT_LAT", "POINT_BALANCE"]

target = 'IT'

glaciers_to_plot = {
    "Pooled": {
        "df": df_src_pooled,
    },
    target: {
        "df": monthly_cache[target]["data_monthly"],
    },
}

fig = plot_kde_pair_reg(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
)

fig.savefig(f"figures/paperTF/appendix/app_kde_Pooled_{target}.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
] + ["POINT_LON", "POINT_LAT", "POINT_BALANCE"]

target = 'ALA'

glaciers_to_plot = {
    "Pooled": {
        "df": df_src_pooled,
    },
    target: {
        "df": monthly_cache[target]["data_monthly"],
    },
}

fig = plot_kde_pair_reg(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
)

fig.savefig(f"figures/paperTF/appendix/app_kde_Pooled_{target}.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
] + ["POINT_LON", "POINT_LAT", "POINT_BALANCE"]

target = 'CA_4'

glaciers_to_plot = {
    "Pooled": {
        "df": df_src_pooled,
    },
    target: {
        "df": monthly_cache[target]["data_monthly"],
    },
}

fig = plot_kde_pair_reg(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
)

fig.savefig(f"figures/paperTF/appendix/app_kde_Pooled_{target}.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
] + ["POINT_LON", "POINT_LAT", "POINT_BALANCE"]

target = 'ALA_4'

glaciers_to_plot = {
    "Pooled": {
        "df": df_src_pooled,
    },
    target: {
        "df": monthly_cache[target]["data_monthly"],
    },
}

fig = plot_kde_pair_reg(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
)

fig.savefig(f"figures/paperTF/appendix/app_kde_Pooled_{target}.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()